# Reproducible results: which neighborhoods complain, and which ones have violations

This notebook recomputes every number in the article on how DOB complaints, ECB
citations, scheduled-inspection violations, and inspection hit rates vary with
census-tract demographics. Click a number in the article and you land on the cell
that produces it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michaelzoorob/dob-complaints-analysis/blob/main/notebooks/neighborhood_results.ipynb)

**How to read this.** Everything is refit here from the committed property panel and
two committed extracts (per-lot caller, agency, and DOB-violation-record counts; a
complaint-level file with outcome and inspector). The producing script is
`scripts/neighborhood_gradients.py`; a final cell cross-checks every refit against
the committed CSVs in `data/analysis/risk_models/`.


In [1]:
# === setup: versions, committed panel and extracts, analysis frame, helpers ===
import sys, warnings
from pathlib import Path
import gc
import numpy as np, pandas as pd, pyfixest as pf
import scipy
warnings.filterwarnings("ignore")
print("python", sys.version.split()[0], "| pyfixest", pf.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)

CAND = [Path.cwd(), Path.cwd().parent] + list(Path.cwd().parents)
ROOT = next((p for p in CAND if (p / "data" / "analysis").exists()), Path.cwd())
DATA = ROOT / "data" / "analysis"
RM = DATA / "risk_models"
PANEL = DATA / "property_risk_panel_v2.csv.gz"
LOT_EXTRACT = DATA / "neighborhood_lot_outcomes.csv.gz"
CPL_EXTRACT = DATA / "neighborhood_complaints.csv.gz"

DTYPE = {"bct2020": str, "size_bin": str, "borocode": str, "bbl_key": str}
WINDOW = ('2020-01-01', '2026-05-31')
YEARS_C = 6.416666666666667
YEARS_E = 6.25
FE_B = 'size_bin + comm_bin + borocode'
VCOV = {'CRV1': 'bct2020'}
MIN_INSPECTOR_CASES = 30
DEMOS = {'tract_poverty10': 'Tract poverty rate (+10 pp)', 'tract_log_income_z': 'Tract log median income (+1 SD)', 'tract_renter10': 'Tract renter share (+10 pp)', 'tract_foreign10': 'Tract foreign-born share (+10 pp)', 'tract_overcrowd10': 'Tract overcrowded-household share (+10 pp)', 'tract_black10': 'Tract Black share (+10 pp)', 'tract_hispanic10': 'Tract Hispanic share (+10 pp)', 'tract_asian10': 'Tract Asian share (+10 pp)'}
GROUPS = {'era': ['era_pre1940', 'era_4079', 'era_8099', 'era_unknown'], 'value': ['value_rank_tract'], 'ownership': ['llc', 'corp_other', 'trust_estate', 'nycha', 'govt', 'owner_occ_star', 'is_coop', 'is_condo', 'geo_nyc_other', 'geo_outside_nyc', 'geo_unknown', 'multi_prop_owner'], 'use_size': ['com_class', 'log_bldgarea', 'log2_area_per_unit', 'mzone', 'multi_bldg'], 'history': ['any_prior_viol']}
CONTROLS = [c for g in GROUPS.values() for c in g]
X = " + ".join(CONTROLS)
OUTCOMES = {'n_caller': 'caller complaints', 'n_agency': 'agency-initiated complaints', 'n_ecb_2020on': 'ECB citations', 'n_dobviol_2020on': 'DOB violation records (scheduled inspections)'}

def build_frame() -> pd.DataFrame:
    df = pd.read_csv(PANEL, dtype=DTYPE, low_memory=False)
    for t in ["llc", "corp_other", "trust_estate", "nycha", "govt"]:
        df[t] = (df["owner_type"] == t).astype(int)
    df = df[df["owner_type"] != "missing"].copy()
    for b in ["owner_occ_star", "is_coop", "is_condo"]:
        df[b] = df[b].astype(int)
    yb = df["yearbuilt"]
    df["era_pre1940"] = yb.between(1800, 1939).astype(int)
    df["era_4079"] = yb.between(1940, 1979).astype(int)
    df["era_8099"] = yb.between(1980, 1999).astype(int)
    df["era_unknown"] = (~yb.between(1800, 2026)).astype(int)
    df["multi_bldg"] = (df["numbldgs"] >= 2).astype(int)
    df["log2_area_per_unit"] = np.log2(df["area_per_unit"])
    ut = pd.to_numeric(df["unitstotal"], errors="coerce")
    ur = pd.to_numeric(df["unitsres"], errors="coerce")
    df["unitscom"] = np.maximum(ut - ur, 0).fillna(0.0)
    cb = [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 25, 50, 100, 250, 100000]
    cl = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
          "11-15", "16-25", "26-50", "51-100", "101-250", "251+"]
    df["comm_bin"] = pd.cut(df["unitscom"], bins=cb, labels=cl).astype(str)
    ba = pd.to_numeric(df["bldgarea"], errors="coerce")
    df["log_bldgarea"] = np.log(ba.where(ba > 0))
    df["com_class"] = df["bldgclass"].astype(str).str[0].isin(["S", "K", "O"]).astype(int)
    for g in ["nyc_other", "outside_nyc", "unknown"]:
        df[f"geo_{g}"] = (df["owner_geo"] == g).astype(int)
    df["multi_prop_owner"] = df["multi_prop_owner"].astype(int)
    df["mzone"] = pd.to_numeric(df["mzone"], errors="coerce").fillna(0).astype(int)
    # assessed value per unit ranked WITHIN the tract, so the value control measures a
    # building's standing relative to its neighbors and cannot absorb between-tract
    # income differences by construction
    vpu = (df["assesstot"] / df["unitsres"]).where((df["assesstot"] > 0) & (df["unitsres"] > 0))
    df["value_rank_tract"] = vpu.groupby(df["bct2020"]).rank(pct=True)
    # demographics: shares 0-1 in the panel -> per +10 pp; log income -> z
    for share in ["tract_poverty", "tract_renter_share", "tract_foreign_born", "tract_overcrowd",
                  "tract_pct_black", "tract_pct_hispanic", "tract_pct_asian"]:
        assert df[share].dropna().between(0, 1).all(), share
    df["tract_poverty10"] = df["tract_poverty"] * 10
    df["tract_renter10"] = df["tract_renter_share"] * 10
    df["tract_foreign10"] = df["tract_foreign_born"] * 10
    df["tract_overcrowd10"] = df["tract_overcrowd"] * 10
    df["tract_black10"] = df["tract_pct_black"] * 10
    df["tract_hispanic10"] = df["tract_pct_hispanic"] * 10
    df["tract_asian10"] = df["tract_pct_asian"] * 10
    li = df["tract_log_income"]
    df["tract_log_income_z"] = (li - li.mean()) / li.std()
    need = ["log2_area_per_unit", "value_rank_tract", "size_bin", "bct2020"] + list(DEMOS)
    keep = df[need].notna().all(axis=1) & np.isfinite(df["log2_area_per_unit"])
    return df[keep].copy()

def irr_row(m, term):
    b = float(m.coef()[term]); se = float(m.se()[term]); lo, hi = m.confint().loc[term].values
    return dict(estimate=b, std_error=se, ci_lo=float(lo), ci_hi=float(hi),
                pct_change=(np.exp(b) - 1) * 100, pct_lo=(np.exp(lo) - 1) * 100,
                pct_hi=(np.exp(hi) - 1) * 100, p=float(m.pvalue()[term]), n=int(m._N))

def pp_row(m, term):
    b = float(m.coef()[term]); se = float(m.se()[term]); lo, hi = m.confint().loc[term].values
    return dict(estimate=b, std_error=se, ci_lo=float(lo), ci_hi=float(hi),
                p=float(m.pvalue()[term]), n=int(m._N))


def committed(csv):
    return pd.read_csv(RM / csv)

lots = pd.read_csv(LOT_EXTRACT, dtype={"bbl_key": str})
cpl = pd.read_csv(CPL_EXTRACT, dtype={"bbl_key": str, "inspector_badge": str, "complaint_category": str})
frame = build_frame().merge(lots, on="bbl_key", how="left")
for c in ["n_caller", "n_agency", "n_dobviol_2020on"]:
    frame[c] = frame[c].fillna(0).astype(int)
frame["any_caller100"] = (frame["n_caller"] > 0).astype(float) * 100
est = frame.dropna(subset=CONTROLS + list(OUTCOMES)).copy()

# complaint-level frames (caller complaints only): all with an outcome, and accessed
keep_cols = ["bbl_key", "size_bin", "comm_bin", "borocode", "bct2020"] + list(DEMOS) + CONTROLS
c_all = cpl.merge(est[keep_cols], on="bbl_key", how="inner")
c_all = c_all[(c_all["caller"] == 1) & c_all["outcome"].isin(["violation", "no_violation", "no_access"])].copy()
c_all["noaccess100"] = (c_all["outcome"] == "no_access").astype(float) * 100
acc = c_all[c_all["outcome"] != "no_access"].copy()
acc["viol100"] = (acc["outcome"] == "violation").astype(float) * 100
acc["inspector_badge"] = acc["inspector_badge"].fillna("").astype(str).str.strip()
_counts = acc["inspector_badge"].value_counts()
acc = acc[acc["inspector_badge"].isin(_counts[_counts >= MIN_INSPECTOR_CASES].index.difference([""]))].copy()
FE_H0 = "complaint_category + size_bin + comm_bin + borocode"
FE_H1 = FE_H0 + " + inspector_badge"
REFIT = {}
print(f"estimation frame: {len(est):,} lots | caller complaints with an outcome: {len(c_all):,} | accessed, inspectors with >= {MIN_INSPECTOR_CASES} cases: {len(acc):,}")

python 3.14.2 | pyfixest 0.60.0 | pandas 3.0.0 | numpy 2.3.5 | scipy 1.17.0


estimation frame: 760,431 lots | caller complaints with an outcome: 391,375 | accessed, inspectors with >= 30 cases: 273,845


## Neighborhood panel and coverage

Sample sizes behind the neighborhood post: lots with tract demographics and building controls, caller versus agency-initiated complaints (a complaint is caller-initiated when it carries a 311 reference number, as in `post0_descriptive_stats.py`), and the complaint-level hit-rate sample (`neighborhood_gradients.py`).

In [2]:
print(f"RESULT residential lots in the estimation frame: {len(est):,}")
print(f"RESULT census tracts represented: {est['bct2020'].nunique():,}")
n_c, n_a = int(cpl["caller"].sum()), int((cpl["caller"] == 0).sum())
print(f"RESULT complaints in window with a lot: {len(cpl):,}  caller {n_c:,} ({n_c/len(cpl)*100:.1f}%)  agency-initiated {n_a:,} ({n_a/len(cpl)*100:.1f}%)")
print(f"RESULT caller complaints per 100 lots per year: {est['n_caller'].sum()/len(est)/YEARS_C*100:.1f}")
print(f"RESULT lots with any caller complaint, 2020 - May 2026: {(est['n_caller']>0).mean()*100:.0f}%")
print(f"RESULT accessed caller inspections in the hit-rate sample: {len(acc):,} across {acc['inspector_badge'].nunique():,} inspectors")
print(f"RESULT baseline hit rate (violation found | accessed): {acc['viol100'].mean():.1f}%   baseline no-access rate: {c_all['noaccess100'].mean():.1f}%")
d = committed("neighborhood_descriptives.csv"); a = d[d.variable == "all"].iloc[0]
assert abs(a["caller_per100_yr"] - est['n_caller'].sum()/len(est)/YEARS_C*100) < 0.05

RESULT residential lots in the estimation frame: 760,431
RESULT census tracts represented: 2,185
RESULT complaints in window with a lot: 783,206  caller 533,247 (68.1%)  agency-initiated 249,959 (31.9%)
RESULT caller complaints per 100 lots per year: 9.3
RESULT lots with any caller complaint, 2020 - May 2026: 20%
RESULT accessed caller inspections in the hit-rate sample: 273,845 across 444 inspectors
RESULT baseline hit rate (violation found | accessed): 27.7%   baseline no-access rate: 29.6%


## Raw rates by tract quintile

Lot-weighted quintiles of tract poverty, foreign-born share, and median income; caller complaints, ECB citations, and DOB violation records per 100 lots per year, plus the hit rate and no-access rate among caller complaints, before any adjustment (`neighborhood_descriptives.csv`).

In [3]:
d = committed("neighborhood_descriptives.csv")
def quintile_table(raw):
    q = pd.qcut(est[raw].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    cq = c_all.merge(pd.DataFrame({"bbl_key": est["bbl_key"], "_q": q.values}), on="bbl_key")
    rows = []
    for k in range(1, 6):
        g = est[q.values == k]; gc = cq[cq["_q"] == k]; ak = gc[gc["outcome"] != "no_access"]
        rows.append(dict(quintile=k, lo=g[raw].min(), hi=g[raw].max(), n_lots=len(g),
                         caller_per100_yr=g["n_caller"].sum()/len(g)/YEARS_C*100,
                         ecb_per100_yr=g["n_ecb_2020on"].sum()/len(g)/YEARS_E*100,
                         dobviol_per100_yr=g["n_dobviol_2020on"].sum()/len(g)/YEARS_C*100,
                         hit_rate=(ak["outcome"] == "violation").mean(), noaccess_rate=(gc["outcome"] == "no_access").mean()))
    return pd.DataFrame(rows)
for var, raw in [("poverty", "tract_poverty"), ("foreign_born", "tract_foreign_born"), ("income", "med_income")]:
    t = quintile_table(raw); lo, hi = t.iloc[0], t.iloc[-1]
    print(f"RESULT {var}: bottom quintile caller complaints {lo.caller_per100_yr:.1f} vs top quintile {hi.caller_per100_yr:.1f} per 100 lots/yr; "
          f"DOB violation records {lo.dobviol_per100_yr:.1f} vs {hi.dobviol_per100_yr:.1f}; hit rate {lo.hit_rate*100:.1f}% vs {hi.hit_rate*100:.1f}%; "
          f"no access {lo.noaccess_rate*100:.1f}% vs {hi.noaccess_rate*100:.1f}%")
    cm = d[d.variable == var].sort_values("quintile")
    assert np.allclose(cm["caller_per100_yr"].values, t["caller_per100_yr"].values, atol=0.05)
    print(t.round(3).to_string(index=False)); print()

RESULT poverty: bottom quintile caller complaints 6.0 vs top quintile 14.5 per 100 lots/yr; DOB violation records 5.9 vs 16.5; hit rate 23.1% vs 31.4%; no access 28.1% vs 25.3%
 quintile    lo    hi  n_lots  caller_per100_yr  ecb_per100_yr  dobviol_per100_yr  hit_rate  noaccess_rate
        1 0.000 0.056  152087             6.044          3.347              5.883     0.231          0.281
        2 0.056 0.091  152086             7.304          3.816              6.545     0.244          0.306
        3 0.091 0.129  152086             8.733          4.618              6.972     0.267          0.355
        4 0.129 0.195  152086            10.143          5.981              9.236     0.282          0.306
        5 0.195 0.770  152086            14.457         10.238             16.469     0.314          0.253



RESULT foreign_born: bottom quintile caller complaints 9.5 vs top quintile 9.1 per 100 lots/yr; DOB violation records 11.7 vs 6.5; hit rate 24.1% vs 29.7%; no access 22.1% vs 41.7%
 quintile    lo    hi  n_lots  caller_per100_yr  ecb_per100_yr  dobviol_per100_yr  hit_rate  noaccess_rate
        1 0.014 0.224  152087             9.480          6.176             11.719     0.241          0.221
        2 0.224 0.328  152086            10.625          6.928             11.937     0.266          0.245
        3 0.328 0.417  152086             8.915          5.525              8.269     0.293          0.290
        4 0.417 0.520  152086             8.546          4.885              6.674     0.300          0.318
        5 0.520 0.844  152086             9.117          4.486              6.507     0.297          0.417



RESULT income: bottom quintile caller complaints 14.1 vs top quintile 9.5 per 100 lots/yr; DOB violation records 15.6 vs 11.7; hit rate 32.2% vs 22.0%; no access 25.3% vs 22.9%
 quintile       lo       hi  n_lots  caller_per100_yr  ecb_per100_yr  dobviol_per100_yr  hit_rate  noaccess_rate
        1  11612.0  66161.0  152087            14.064          9.677             15.608     0.322          0.253
        2  66161.0  83351.0  152086             9.653          5.238              8.016     0.285          0.337
        3  83351.0  98603.0  152086             7.203          3.869              5.754     0.271          0.347
        4  98603.0 117832.0  152086             6.247          3.035              4.039     0.261          0.366
        5 117832.0 250001.0  152086             9.516          6.180             11.689     0.220          0.229



## Caller complaint gradients by tract demographics

Poisson pseudo-maximum-likelihood models of caller complaints per lot on each tract demographic, one at a time. `total` holds building size fixed only (unit-count, commercial-unit, and borough fixed effects); `direct` adds the building-stock controls (era, within-tract value rank, ownership, use and size detail, prior violations). Standard errors clustered by tract (`neighborhood_gradients.py`, `neighborhood_gradients.csv`).

In [4]:
y = "n_caller"
for d, dlab in DEMOS.items():
    out = {}
    for spec, rhs in [("total", d), ("direct", f"{d} + {X}")]:
        m = pf.fepois(f"{y} ~ {rhs} | {FE_B}", data=est, vcov=VCOV, lean=True, store_data=False, copy_data=False)
        r = irr_row(m, d); out[spec] = r; REFIT[(y, spec, d)] = r["pct_change"]
    t, dr = out["total"], out["direct"]
    print(f"RESULT {dlab}: total {t['pct_change']:+.1f}% [{t['pct_lo']:+.1f}, {t['pct_hi']:+.1f}]   "
          f"direct {dr['pct_change']:+.1f}% [{dr['pct_lo']:+.1f}, {dr['pct_hi']:+.1f}]   N={t['n']:,}")
    del m; gc.collect()

RESULT Tract poverty rate (+10 pp): total +8.5% [+6.6, +10.5]   direct +5.4% [+3.6, +7.3]   N=760,431


RESULT Tract log median income (+1 SD): total -7.5% [-9.0, -6.0]   direct -5.7% [-7.3, -4.1]   N=760,431


RESULT Tract renter share (+10 pp): total +5.0% [+3.8, +6.2]   direct +2.1% [+0.9, +3.2]   N=760,431


RESULT Tract foreign-born share (+10 pp): total +4.7% [+2.9, +6.5]   direct +4.3% [+2.5, +6.1]   N=760,431


RESULT Tract overcrowded-household share (+10 pp): total +8.2% [+5.3, +11.3]   direct +5.3% [+2.6, +8.1]   N=760,431


RESULT Tract Black share (+10 pp): total -0.4% [-1.2, +0.4]   direct -0.9% [-1.7, -0.2]   N=760,431


RESULT Tract Hispanic share (+10 pp): total +3.6% [+2.6, +4.6]   direct +1.8% [+0.8, +2.7]   N=760,431


RESULT Tract Asian share (+10 pp): total +2.4% [+1.2, +3.6]   direct +3.3% [+2.2, +4.4]   N=760,431


## ECB citation gradients by tract demographics

Poisson pseudo-maximum-likelihood models of ECB citations per lot on each tract demographic, one at a time. `total` holds building size fixed only (unit-count, commercial-unit, and borough fixed effects); `direct` adds the building-stock controls (era, within-tract value rank, ownership, use and size detail, prior violations). Standard errors clustered by tract (`neighborhood_gradients.py`, `neighborhood_gradients.csv`).

In [5]:
y = "n_ecb_2020on"
for d, dlab in DEMOS.items():
    out = {}
    for spec, rhs in [("total", d), ("direct", f"{d} + {X}")]:
        m = pf.fepois(f"{y} ~ {rhs} | {FE_B}", data=est, vcov=VCOV, lean=True, store_data=False, copy_data=False)
        r = irr_row(m, d); out[spec] = r; REFIT[(y, spec, d)] = r["pct_change"]
    t, dr = out["total"], out["direct"]
    print(f"RESULT {dlab}: total {t['pct_change']:+.1f}% [{t['pct_lo']:+.1f}, {t['pct_hi']:+.1f}]   "
          f"direct {dr['pct_change']:+.1f}% [{dr['pct_lo']:+.1f}, {dr['pct_hi']:+.1f}]   N={t['n']:,}")
    del m; gc.collect()

RESULT Tract poverty rate (+10 pp): total +12.2% [+10.3, +14.2]   direct +9.7% [+7.9, +11.5]   N=760,431


RESULT Tract log median income (+1 SD): total -7.8% [-9.2, -6.4]   direct -7.5% [-8.9, -6.1]   N=760,431


RESULT Tract renter share (+10 pp): total +7.8% [+6.8, +8.9]   direct +4.3% [+3.3, +5.3]   N=760,431


RESULT Tract foreign-born share (+10 pp): total +2.6% [+1.2, +4.1]   direct +3.4% [+2.0, +4.8]   N=760,431


RESULT Tract overcrowded-household share (+10 pp): total +8.7% [+6.2, +11.2]   direct +5.8% [+3.5, +8.2]   N=760,431


RESULT Tract Black share (+10 pp): total +2.2% [+1.4, +3.0]   direct +2.1% [+1.4, +2.9]   N=760,431


RESULT Tract Hispanic share (+10 pp): total +3.6% [+2.5, +4.7]   direct +1.9% [+0.9, +3.0]   N=760,431


RESULT Tract Asian share (+10 pp): total -0.8% [-2.1, +0.6]   direct +0.8% [-0.5, +2.0]   N=760,431


## DOB violation record gradients by tract demographics

Poisson pseudo-maximum-likelihood models of DOB violation records per lot on each tract demographic, one at a time. `total` holds building size fixed only (unit-count, commercial-unit, and borough fixed effects); `direct` adds the building-stock controls (era, within-tract value rank, ownership, use and size detail, prior violations). Standard errors clustered by tract (`neighborhood_gradients.py`, `neighborhood_gradients.csv`).

In [6]:
y = "n_dobviol_2020on"
for d, dlab in DEMOS.items():
    out = {}
    for spec, rhs in [("total", d), ("direct", f"{d} + {X}")]:
        m = pf.fepois(f"{y} ~ {rhs} | {FE_B}", data=est, vcov=VCOV, lean=True, store_data=False, copy_data=False)
        r = irr_row(m, d); out[spec] = r; REFIT[(y, spec, d)] = r["pct_change"]
    t, dr = out["total"], out["direct"]
    print(f"RESULT {dlab}: total {t['pct_change']:+.1f}% [{t['pct_lo']:+.1f}, {t['pct_hi']:+.1f}]   "
          f"direct {dr['pct_change']:+.1f}% [{dr['pct_lo']:+.1f}, {dr['pct_hi']:+.1f}]   N={t['n']:,}")
    del m; gc.collect()

RESULT Tract poverty rate (+10 pp): total +10.1% [+8.2, +12.0]   direct +5.4% [+4.1, +6.7]   N=760,431


RESULT Tract log median income (+1 SD): total -7.0% [-8.6, -5.3]   direct -5.1% [-6.1, -4.0]   N=760,431


RESULT Tract renter share (+10 pp): total +3.9% [+2.7, +5.2]   direct +2.5% [+1.7, +3.3]   N=760,431


RESULT Tract foreign-born share (+10 pp): total -1.4% [-2.6, -0.1]   direct +1.4% [+0.3, +2.4]   N=760,431


RESULT Tract overcrowded-household share (+10 pp): total +5.2% [+3.0, +7.5]   direct +4.2% [+2.5, +6.0]   N=760,431


RESULT Tract Black share (+10 pp): total +3.0% [+2.3, +3.7]   direct +3.2% [+2.6, +3.7]   N=760,431


RESULT Tract Hispanic share (+10 pp): total +1.3% [+0.3, +2.3]   direct +0.5% [-0.3, +1.2]   N=760,431


RESULT Tract Asian share (+10 pp): total -1.4% [-2.6, -0.1]   direct -0.3% [-1.3, +0.7]   N=760,431


## Agency initiated complaint gradients by tract demographics

Poisson pseudo-maximum-likelihood models of agency-initiated complaints per lot on each tract demographic, one at a time. `total` holds building size fixed only (unit-count, commercial-unit, and borough fixed effects); `direct` adds the building-stock controls (era, within-tract value rank, ownership, use and size detail, prior violations). Standard errors clustered by tract (`neighborhood_gradients.py`, `neighborhood_gradients.csv`).

In [7]:
y = "n_agency"
for d, dlab in DEMOS.items():
    out = {}
    for spec, rhs in [("total", d), ("direct", f"{d} + {X}")]:
        m = pf.fepois(f"{y} ~ {rhs} | {FE_B}", data=est, vcov=VCOV, lean=True, store_data=False, copy_data=False)
        r = irr_row(m, d); out[spec] = r; REFIT[(y, spec, d)] = r["pct_change"]
    t, dr = out["total"], out["direct"]
    print(f"RESULT {dlab}: total {t['pct_change']:+.1f}% [{t['pct_lo']:+.1f}, {t['pct_hi']:+.1f}]   "
          f"direct {dr['pct_change']:+.1f}% [{dr['pct_lo']:+.1f}, {dr['pct_hi']:+.1f}]   N={t['n']:,}")
    del m; gc.collect()

RESULT Tract poverty rate (+10 pp): total +6.4% [+4.3, +8.5]   direct +4.1% [+2.2, +5.9]   N=760,431


RESULT Tract log median income (+1 SD): total -0.3% [-2.1, +1.6]   direct -0.2% [-1.8, +1.5]   N=760,431


RESULT Tract renter share (+10 pp): total +3.8% [+2.6, +5.0]   direct +1.0% [-0.1, +2.0]   N=760,431


RESULT Tract foreign-born share (+10 pp): total -4.3% [-5.8, -2.8]   direct -2.6% [-4.0, -1.1]   N=760,431


RESULT Tract overcrowded-household share (+10 pp): total +1.8% [-0.7, +4.5]   direct +0.0% [-2.2, +2.3]   N=760,431


RESULT Tract Black share (+10 pp): total +0.4% [-0.5, +1.2]   direct +0.5% [-0.2, +1.3]   N=760,431


RESULT Tract Hispanic share (+10 pp): total +0.3% [-0.8, +1.5]   direct -1.0% [-2.0, +0.1]   N=760,431


RESULT Tract Asian share (+10 pp): total -0.8% [-2.1, +0.5]   direct +0.8% [-0.5, +2.1]   N=760,431


## Building stock decomposition of the caller complaint gradient

Linear probability model of any caller complaint (percentage points) on each demographic, with and without the building-stock controls, and the exact Gelbach (2016) decomposition of the gap into control groups: each control's coefficient in the full model times its own gradient on the demographic. The group contributions sum to the gap by construction (`neighborhood_decomposition.csv`).

In [8]:
dec = committed("neighborhood_decomposition.csv")
for d, dlab in DEMOS.items():
    base = pf.feols(f"any_caller100 ~ {d} | {FE_B}", data=est, vcov=VCOV, lean=True, store_data=False, copy_data=False)
    full = pf.feols(f"any_caller100 ~ {d} + {X} | {FE_B}", data=est, vcov=VCOV, lean=True, store_data=False, copy_data=False)
    b0, b1 = float(base.coef()[d]), float(full.coef()[d])
    contrib = {k: float(pf.feols(f"{k} ~ {d} | {FE_B}", data=est, vcov="iid", lean=True, store_data=False, copy_data=False).coef()[d]) * float(full.coef()[k]) for k in CONTROLS}
    assert np.isclose(sum(contrib.values()), b0 - b1, rtol=1e-3, atol=1e-6)
    parts = {g: sum(contrib[k] for k in ks) for g, ks in GROUPS.items()}
    REFIT[("gelbach", "total", d)] = b0; REFIT[("gelbach", "direct", d)] = b1
    for g, v in parts.items(): REFIT[("gelbach", f"via_{g}", d)] = v
    br, fr = pp_row(base, d), pp_row(full, d)
    print(f"RESULT {dlab}: total {b0:+.2f} pp [{br['ci_lo']:+.2f}, {br['ci_hi']:+.2f}]  left over {b1:+.2f} pp [{fr['ci_lo']:+.2f}, {fr['ci_hi']:+.2f}]  "
          + "  ".join(f"via {g} {v:+.2f}" for g, v in parts.items())
          + (f"   (left over = {b1/b0*100:.0f}% of total)" if abs(b0) > 1e-9 else ""))
    del base, full; gc.collect()

RESULT Tract poverty rate (+10 pp): total +1.23 pp [+0.92, +1.53]  left over +0.68 pp [+0.41, +0.95]  via era +0.07  via value +0.04  via ownership +0.30  via use_size +0.00  via history +0.13   (left over = 55% of total)


RESULT Tract log median income (+1 SD): total -0.94 pp [-1.22, -0.67]  left over -0.68 pp [-0.94, -0.41]  via era +0.00  via value -0.04  via ownership -0.22  via use_size +0.05  via history -0.07   (left over = 72% of total)


RESULT Tract renter share (+10 pp): total +0.60 pp [+0.45, +0.74]  left over +0.24 pp [+0.09, +0.39]  via era +0.12  via value +0.04  via ownership +0.12  via use_size +0.00  via history +0.07   (left over = 40% of total)


RESULT Tract foreign-born share (+10 pp): total +1.23 pp [+0.95, +1.51]  left over +1.26 pp [+0.99, +1.52]  via era +0.01  via value +0.01  via ownership -0.03  via use_size -0.06  via history +0.04   (left over = 102% of total)


RESULT Tract overcrowded-household share (+10 pp): total +1.77 pp [+1.25, +2.30]  left over +1.30 pp [+0.81, +1.79]  via era +0.20  via value +0.05  via ownership +0.06  via use_size +0.02  via history +0.15   (left over = 73% of total)


RESULT Tract Black share (+10 pp): total -0.23 pp [-0.33, -0.13]  left over -0.23 pp [-0.32, -0.14]  via era -0.04  via value -0.00  via ownership +0.07  via use_size -0.02  via history -0.01   (left over = 99% of total)


RESULT Tract Hispanic share (+10 pp): total +0.66 pp [+0.52, +0.81]  left over +0.32 pp [+0.18, +0.47]  via era +0.15  via value +0.03  via ownership +0.11  via use_size -0.01  via history +0.06   (left over = 49% of total)


RESULT Tract Asian share (+10 pp): total +0.75 pp [+0.55, +0.94]  left over +0.82 pp [+0.64, +0.99]  via era -0.04  via value -0.00  via ownership -0.05  via use_size +0.00  via history +0.02   (left over = 110% of total)


## Hit rate by tract demographics with inspector fixed effects

Complaint-level linear probability model: violation found (percentage points) among caller complaints where the inspector accessed the property, on each tract demographic. Fixed effects for complaint category, unit-count, commercial-unit, and borough; then adding inspector fixed effects (same inspector, complaints from different tracts); then adding the building-stock controls. Inspectors with at least 30 accessed caller inspections. Standard errors clustered by tract (`neighborhood_hitrate.csv`).

In [9]:
print(f"baseline hit rate: {acc['viol100'].mean():.1f}%  N={len(acc):,}")
for d, dlab in DEMOS.items():
    out = {}
    for spec, rhs, fe in [("hit_base", d, FE_H0), ("hit_inspector_fe", d, FE_H1), ("hit_inspector_fe_controls", f"{d} + {X}", FE_H1)]:
        m = pf.feols(f"viol100 ~ {rhs} | {fe}", data=acc, vcov=VCOV, lean=True, store_data=False, copy_data=False)
        r = pp_row(m, d); out[spec] = r; REFIT[("hit", spec, d)] = r["estimate"]
    print(f"RESULT {dlab}: base {out['hit_base']['estimate']:+.2f} pp   with inspector FE {out['hit_inspector_fe']['estimate']:+.2f} pp "
          f"[{out['hit_inspector_fe']['ci_lo']:+.2f}, {out['hit_inspector_fe']['ci_hi']:+.2f}]   "
          f"plus building controls {out['hit_inspector_fe_controls']['estimate']:+.2f} pp [{out['hit_inspector_fe_controls']['ci_lo']:+.2f}, {out['hit_inspector_fe_controls']['ci_hi']:+.2f}]")
    del m; gc.collect()

baseline hit rate: 27.7%  N=273,845


RESULT Tract poverty rate (+10 pp): base +1.05 pp   with inspector FE +1.03 pp [+0.76, +1.31]   plus building controls +1.07 pp [+0.79, +1.35]


RESULT Tract log median income (+1 SD): base -1.16 pp   with inspector FE -1.06 pp [-1.28, -0.83]   plus building controls -1.15 pp [-1.39, -0.92]


RESULT Tract renter share (+10 pp): base +0.01 pp   with inspector FE +0.34 pp [+0.20, +0.49]   plus building controls +0.34 pp [+0.19, +0.49]


RESULT Tract foreign-born share (+10 pp): base +0.64 pp   with inspector FE +0.53 pp [+0.33, +0.73]   plus building controls +0.53 pp [+0.32, +0.73]


RESULT Tract overcrowded-household share (+10 pp): base +1.07 pp   with inspector FE +1.20 pp [+0.80, +1.60]   plus building controls +1.21 pp [+0.81, +1.60]


RESULT Tract Black share (+10 pp): base +0.61 pp   with inspector FE +0.44 pp [+0.31, +0.57]   plus building controls +0.46 pp [+0.33, +0.59]


RESULT Tract Hispanic share (+10 pp): base +0.05 pp   with inspector FE +0.27 pp [+0.11, +0.43]   plus building controls +0.22 pp [+0.06, +0.37]


RESULT Tract Asian share (+10 pp): base -0.44 pp   with inspector FE -0.29 pp [-0.48, -0.10]   plus building controls -0.25 pp [-0.43, -0.06]


## No access by tract demographics

Complaint-level linear probability model: the inspection ended without access (percentage points) among caller complaints with an outcome, on each tract demographic, with and without the building-stock controls; category, unit-count, commercial-unit, and borough fixed effects, standard errors clustered by tract (`neighborhood_hitrate.csv`).

In [10]:
print(f"baseline no-access rate: {c_all['noaccess100'].mean():.1f}%  N={len(c_all):,}")
for d, dlab in DEMOS.items():
    out = {}
    for spec, rhs in [("noaccess_base", d), ("noaccess_controls", f"{d} + {X}")]:
        m = pf.feols(f"noaccess100 ~ {rhs} | {FE_H0}", data=c_all, vcov=VCOV, lean=True, store_data=False, copy_data=False)
        r = pp_row(m, d); out[spec] = r; REFIT[("noaccess", spec, d)] = r["estimate"]
    print(f"RESULT {dlab}: base {out['noaccess_base']['estimate']:+.2f} pp [{out['noaccess_base']['ci_lo']:+.2f}, {out['noaccess_base']['ci_hi']:+.2f}]   "
          f"with building controls {out['noaccess_controls']['estimate']:+.2f} pp [{out['noaccess_controls']['ci_lo']:+.2f}, {out['noaccess_controls']['ci_hi']:+.2f}]")
    del m; gc.collect()

baseline no-access rate: 29.6%  N=391,375


RESULT Tract poverty rate (+10 pp): base +0.60 pp [+0.36, +0.84]   with building controls +0.53 pp [+0.30, +0.77]


RESULT Tract log median income (+1 SD): base -0.48 pp [-0.71, -0.26]   with building controls -0.44 pp [-0.67, -0.22]


RESULT Tract renter share (+10 pp): base +0.18 pp [+0.03, +0.34]   with building controls +0.18 pp [+0.03, +0.33]


RESULT Tract foreign-born share (+10 pp): base +0.53 pp [+0.31, +0.75]   with building controls +0.46 pp [+0.25, +0.67]


RESULT Tract overcrowded-household share (+10 pp): base +1.12 pp [+0.77, +1.47]   with building controls +0.97 pp [+0.64, +1.31]


RESULT Tract Black share (+10 pp): base -0.19 pp [-0.29, -0.08]   with building controls -0.21 pp [-0.31, -0.11]


RESULT Tract Hispanic share (+10 pp): base +0.40 pp [+0.27, +0.54]   with building controls +0.33 pp [+0.20, +0.46]


RESULT Tract Asian share (+10 pp): base +0.33 pp [+0.15, +0.51]   with building controls +0.33 pp [+0.15, +0.50]


## Verification against committed estimates

Every refit above compared with the committed CSVs written by `neighborhood_gradients.py`. Refits use the same committed data and code, so they should agree to rounding.

In [11]:
g = committed("neighborhood_gradients.csv"); g = g[g.entry == "bivariate"]
h = committed("neighborhood_hitrate.csv"); dec = committed("neighborhood_decomposition.csv")
rows = []
for (kind, spec, d), v in REFIT.items():
    if kind in OUTCOMES:
        c = float(g[(g.outcome == kind) & (g.spec == spec) & (g.term == d)]["pct_change"].iloc[0]); tol = 0.05
    elif kind == "gelbach":
        c = float(dec[(dec.term == d) & (dec.component == spec)]["points"].iloc[0]); tol = 0.005
    else:
        c = float(h[(h.spec == spec) & (h.term == d)]["estimate"].iloc[0]); tol = 0.005
    rows.append({"result": f"{kind} / {spec} / {d}", "refit": round(v, 3), "committed": round(c, 3),
                 "check": "PASS" if abs(v - c) <= tol else "REVIEW"})
out = pd.DataFrame(rows)
print(out.to_string(index=False))
print(f"\n{(out.check == 'PASS').sum()} of {len(out)} PASS")
assert (out.check == "PASS").all(), out[out.check != "PASS"]

                                              result  refit  committed check
                  n_caller / total / tract_poverty10  8.538      8.538  PASS
                 n_caller / direct / tract_poverty10  5.428      5.428  PASS
               n_caller / total / tract_log_income_z -7.549     -7.549  PASS
              n_caller / direct / tract_log_income_z -5.729     -5.729  PASS
                   n_caller / total / tract_renter10  4.995      4.995  PASS
                  n_caller / direct / tract_renter10  2.065      2.065  PASS
                  n_caller / total / tract_foreign10  4.694      4.694  PASS
                 n_caller / direct / tract_foreign10  4.285      4.285  PASS
                n_caller / total / tract_overcrowd10  8.247      8.247  PASS
               n_caller / direct / tract_overcrowd10  5.308      5.308  PASS
                    n_caller / total / tract_black10 -0.442     -0.442  PASS
                   n_caller / direct / tract_black10 -0.926     -0.926  PASS